# BBC News: Sport Category Analysis
## Traditional NLP Pipeline: TF-IDF + LDA Sub-category Discovery


In [1]:
# Core libraries
import os
import re
import sys

# Data handling
import pandas as pd
import numpy as np

# NLP — Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# NLP — Topic Modelling
from sklearn.decomposition import LatentDirichletAllocation

# Display
from rich.console import Console
from rich.table import Table

# Add parent directory to path so we can access loader.py
sys.path.append('..')
from loader import load_data

## Load the Data for Sport only

In [4]:
# Load all articles and labels
documents, labels = load_data('../data')

# Filter sport articles only
sport_docs = [documents[i] for i in range(len(documents)) if labels[i] == 'sport']

print(f"Total articles in dataset: {len(documents)}")
print(f"Sport articles: {len(sport_docs)}")
print(f"\nSample sport article:\n{sport_docs[0][:300]}")

Total articles in dataset: 2225
Sport articles: 511

Sample sport article:
Fuming Robinson blasts officials

England coach Andy Robinson insisted he was "livid" after his side were denied two tries in Sunday's 19-13 Six Nations loss to Ireland in Dublin.

Mark Cueto's first-half effort was ruled out for offside before the referee spurned TV replays when England crashed ove


## Clean the Data


In [5]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Custom stopwords — domain noise specific to BBC News
custom_stopwords = set([
    'said', 'says', 'say', 'mr', 'mrs', 'ms', 'bn', 'year', 
    'years', 'new', 'people', 'also', 'one', 'two', 'three',
    'told', 'added', 'according', 'last', 'first', 'time',
    'week', 'month', 'ago', 'would', 'could', 'may', 'also',
    'fue', 'use', 'used', 'make', 'making', 'made', 'good',
    'best', 'big', 'got', 'get', 'getting'
])

# Combine with sklearn stopwords
all_stopwords = ENGLISH_STOP_WORDS.union(custom_stopwords)

def clean_text(text):
    # Lowercase everything
    text = text.lower()
    
    # Remove text inside square brackets
    text = re.sub(r'\[.*?\]', '', text)
    
    # Remove non-letter characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove stopwords — now using expanded list
    tokens = text.split()
    tokens = [word for word in tokens if word not in all_stopwords]
    
    return ' '.join(tokens)

In [7]:
# Apply cleaning to sport articles
cleaned_sport = [clean_text(doc) for doc in sport_docs]

print(f"Cleaning complete: {len(cleaned_sport)} sport articles cleaned")
print(f"\nBefore cleaning:\n{sport_docs[0][:200]}")
print(f"\nAfter cleaning:\n{cleaned_sport[0][:200]}")


Cleaning complete: 511 sport articles cleaned

Before cleaning:
Fuming Robinson blasts officials

England coach Andy Robinson insisted he was "livid" after his side were denied two tries in Sunday's 19-13 Six Nations loss to Ireland in Dublin.

Mark Cueto's first-

After cleaning:
fuming robinson blasts officials england coach andy robinson insisted livid denied tries sundays nations loss ireland dublin mark cuetos firsthalf effort ruled offside referee spurned tv replays engla


## Remove Duplicates

In [8]:
# Remove duplicates from sport articles
seen = set()
unique_sport = []

for doc in cleaned_sport:
    if doc not in seen:
        seen.add(doc)
        unique_sport.append(doc)

print(f"Before deduplication: {len(cleaned_sport)} articles")
print(f"After deduplication: {len(unique_sport)} articles")
print(f"Duplicates removed: {len(cleaned_sport) - len(unique_sport)}")

Before deduplication: 511 articles
After deduplication: 501 articles
Duplicates removed: 10


## TF-IDF Feature Extraction

In [9]:
# Initialize TF-IDF Vectorizer for business articles
tfidf = TfidfVectorizer(
    max_features=1000,
    max_df=0.95,
    min_df=2
)

# Fit and transform sport articles only
sport_tfidf = tfidf.fit_transform(unique_sport)

print(f"TF-IDF Matrix Shape: {sport_tfidf.shape}")
print(f"\nThis means:")
print(f"  {sport_tfidf.shape[0]} sport articles (rows)")
print(f"  {sport_tfidf.shape[1]} features/words (columns)")

feature_names = tfidf.get_feature_names_out()
print(f"\nTop 20 words in sport vocabulary:")
print(feature_names[:20])

TF-IDF Matrix Shape: (501, 1000)

This means:
  501 sport articles (rows)
  1000 features/words (columns)

Top 20 words in sport vocabulary:
['able' 'absence' 'ac' 'action' 'admitted' 'advantage' 'africa' 'african'
 'agassi' 'agreed' 'ahead' 'alan' 'alex' 'allegations' 'allow' 'almunia'
 'alongside' 'american' 'andre' 'andrew']


## LDA Topic Modelling (Sub-category Discovery)

In [10]:
# Run LDA on sport articles
lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(sport_tfidf)

print("Sport Sub-categories Discovered:")
print("=" * 50)

feature_names = tfidf.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f"\nTopic {topic_idx + 1}:")
    print(f"  Top words: {', '.join(top_words)}")

Sport Sub-categories Discovered:

Topic 1:
  Top words: england, game, cup, wales, players, play, club, rugby, win, ireland

Topic 2:
  Top words: champion, seed, world, race, olympic, set, indoor, championships, european, roddick

Topic 3:
  Top words: drugs, conte, balco, banned, mutu, doping, antidoping, jones, ban, juninho


## Evaluate LDA with Coherence Score

In [11]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Prepare tokenized documents for gensim
tokenized_docs = [doc.split() for doc in unique_sport]

# Build gensim dictionary
gensim_dict = Dictionary(tokenized_docs)

# Get top 10 words per topic as lists
topics_words = []
for topic in lda.components_:
    top_indices = topic.argsort()[:-11:-1]
    top_words = [feature_names[i] for i in top_indices]
    topics_words.append(top_words)

# Calculate coherence score
coherence_model = CoherenceModel(
    topics=topics_words,
    texts=tokenized_docs,
    dictionary=gensim_dict,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()
print(f"LDA Coherence Score (c_v): {coherence_score:.4f}")

# Interpretation
if coherence_score > 0.7:
    print("Excellent — topics are very well defined")
elif coherence_score > 0.5:
    print("Good — topics are clear and meaningful")
elif coherence_score > 0.3:
    print("Moderate — topics are somewhat meaningful")
else:
    print("Poor — consider adjusting number of topics")


LDA Coherence Score (c_v): 0.5146
Good — topics are clear and meaningful
